<!-- 학습 보강 셀 -->

# 07. Semantic Similarity 학습 흐름

이 노트북은 두 문장이 의미적으로 얼마나 비슷한지 임베딩 기반으로 평가합니다.
RAG에서는 답변 평가, 문서 중복 제거, 검색 결과 품질 점검에 이런 유사도 계산을 자주 사용합니다.

In [1]:
# Jupyter 환경에서 비동기 평가 함수를 안전하게 실행하기 위한 설정입니다.
# - 이미 이벤트 루프가 실행 중인 노트북에서 await를 사용할 때 필요합니다.
import nest_asyncio
nest_asyncio.apply()

In [2]:
from llama_index.core.evaluation import SemanticSimilarityEvaluator
from llama_index.embeddings.ollama import OllamaEmbedding

OLLAMA_BASE_URL = 'http://localhost:11434'

# 01번 노트와 동일하게 기존 실습 임베딩 모델인 nomic-embed-text를 사용합니다.
# 다른 임베딩 모델을 쓰려면 아래 값과 FAISS 등 벡터 차원 설정을 함께 확인해야 합니다.
OLLAMA_EMBED_MODEL = 'nomic-embed-text'

# 의미 유사도 평가는 두 문장을 같은 임베딩 공간에 놓고 가까운 정도를 계산합니다.
# 실행 전 `ollama serve`가 필요합니다. 아래 준비 셀에서 모델이 없으면 자동으로 pull 합니다.
embed_model = OllamaEmbedding(
    model_name=OLLAMA_EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

In [3]:
# OLLAMA_MODEL_PREP_CELL
# Ollama 모델은 pip/requirements.txt로 설치되지 않습니다.
# 이 셀은 노트북 실행 전에 필요한 로컬 Ollama 모델이 있는지 확인하고, 없으면 자동으로 pull 합니다.
import subprocess

def _installed_ollama_models() -> set[str]:
    try:
        result = subprocess.run(
            ['ollama', 'list'],
            check=True,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise RuntimeError('Ollama CLI가 설치되어 있지 않습니다. https://ollama.com 에서 설치하세요.') from exc
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Ollama 서버가 실행 중인지 확인하세요. 터미널에서 `ollama serve`를 실행하세요.') from exc

    names = set()
    for line in result.stdout.splitlines()[1:]:
        parts = line.split()
        if parts:
            names.add(parts[0])
    return names

def ensure_ollama_model(model_name: str) -> None:
    installed = _installed_ollama_models()
    candidates = {model_name}
    if ':' not in model_name:
        candidates.add(f'{model_name}:latest')

    if installed.intersection(candidates):
        print(f'이미 설치됨: {model_name}')
        return

    print(f'Ollama 모델 다운로드 중: {model_name}')
    subprocess.run(['ollama', 'pull', model_name], check=True)

for model_name in [OLLAMA_EMBED_MODEL]:
    ensure_ollama_model(model_name)


이미 설치됨: nomic-embed-text


<!-- 학습 보강 셀 -->

## 의미 유사도와 문자열 유사도의 차이

문자열 유사도는 같은 단어가 얼마나 겹치는지를 보지만, 의미 유사도는 표현이 달라도 같은 뜻인지에 초점을 둡니다.
예를 들어 “스트레스 완화”와 “불안 감소”는 단어가 완전히 같지 않아도 의미적으로 가까울 수 있습니다.

In [4]:
# similarity_threshold 이상이면 passing=True로 판단합니다.
# 임계값은 데이터와 목적에 맞게 조정해야 하며, 0.5는 실습용으로 낮게 잡은 값입니다.
evaluator = SemanticSimilarityEvaluator(
    embed_model=embed_model,
    similarity_threshold=0.5,
)

<!-- 학습 보강 셀 -->

## threshold는 정답이 아니라 기준값

`similarity_threshold`는 데이터와 목적에 맞게 조정해야 합니다.
너무 낮으면 관련 없는 문장도 통과하고, 너무 높으면 표현이 다른 정답 문장까지 탈락할 수 있습니다.
실제 평가에서는 여러 예시를 보고 기준값을 경험적으로 정합니다.

In [5]:
# 평가할 문장들
# - sentence1은 요가의 효과를 설명하는 기준 문장입니다.
# - sentence2는 같은 의미를 다른 표현으로 말한 문장입니다.
sentence1 = """요가는 신체와 마음 모두에게 다양한 이점을 제공합니다. 유연성, 근력, 균형감을 향상시키는 동시에 스트레스, 불안, 통증을 줄여줍니다. 요가는 또한 숙면, 심장 건강, 전반적인 웰빙을 증진시킵니다. 운동이나 스트레스 해소 방법을 찾고 있다면, 요가는 좋은 선택이 될 수 있습니다."""

sentence2 = """요가는 몸과 마음 건강을 함께 돕는 운동입니다. 꾸준히 하면 유연성과 근력이 좋아지고 스트레스 완화에도 도움이 됩니다. 또한 수면과 심장 건강에도 긍정적인 영향을 줄 수 있습니다."""

In [6]:
# 동기 방식 평가
result = evaluator.evaluate(
    response=sentence1,
    reference=sentence2,
)

<!-- 학습 보강 셀 -->

## 동기 평가와 비동기 평가

동기 방식은 코드 흐름이 단순해서 이해하기 좋고, 비동기 방식은 여러 평가를 동시에 처리할 때 유리합니다.
노트북 학습 단계에서는 두 방식이 같은 목적의 API라는 점만 이해하면 됩니다.

In [7]:
# 비동기 방식 평가
# - 동기 평가를 먼저 실행하면 OllamaEmbedding 내부 async client가 닫힌 이벤트 루프를 참조할 수 있습니다.
# - 비동기 예제에서는 현재 노트북 이벤트 루프에 맞는 evaluator를 새로 만듭니다.
async_embed_model = OllamaEmbedding(
    model_name=OLLAMA_EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)
async_evaluator = SemanticSimilarityEvaluator(
    embed_model=async_embed_model,
    similarity_threshold=0.5,
)

result = await async_evaluator.aevaluate(
    response=sentence1,
    reference=sentence2,
)

In [8]:
# 평가 결과 확인
# - score: 두 문장의 의미적 유사도
# - passing: score가 threshold 이상인지 여부
print('유사도 점수:', result.score)
print('통과 여부:', result.passing)

유사도 점수: 0.9063735110290556
통과 여부: True
